# Function 6

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.78287982 0

Here we add the data provided with the weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5, 0.5], [0.017322, 0.062982, 0.093079, 0.018967, 0.059868], [0.845849, 0.032354, 0.981185, 0.024732, 0.947127]]
additionalOutputs = [np.float64(-1.0738405486176665), np.float64(-2.048147931140859), np.float64(-2.586900638160862)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.78287982 0

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [4]:
#Initialise query lists and maximum observations
X, Y = input, output

# Latin Hypercube Sampling (LHS) ---
n_samples = 12500000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(12500000, 5)


In [5]:
print(x_grid[:10])

[[0.58026194 0.8737674  0.45546705 0.03321106 0.87279239]
 [0.9409424  0.18936754 0.64474858 0.30689231 0.66127068]
 [0.03917549 0.15294017 0.76725947 0.52597409 0.7291894 ]
 [0.91680358 0.97023604 0.79387959 0.49314577 0.59302387]
 [0.96027354 0.84575429 0.85817288 0.20734745 0.86575738]
 [0.0186691  0.88265332 0.95137248 0.43581895 0.09256867]
 [0.86290234 0.67066784 0.66988725 0.43072741 0.4402478 ]
 [0.48798566 0.92678735 0.99398364 0.59283518 0.43129419]
 [0.53505949 0.52454161 0.40608674 0.38223734 0.64701809]
 [0.87231898 0.61308397 0.25468734 0.81418035 0.31444223]]


# Bayesian Optimisation with UCB applied

In [6]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
beta = 1.96
#beta = 0.5

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.95767735 0.01826954 0.0535562  0.01930542 0.98302031]
